# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Msdff/FlyRankAiAssignment/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

In [ ]:
import os
print(os.getcwd())
os.chdir("..")   

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# Rebuild baseline (Week 4)
stale = (df["days_since_last_update"] >= 180).astype(int)
visible = (df["impressions_90d"] >= 500).astype(int)
df["baseline_score"] = stale * visible * df["impressions_90d"]

# Rebuild model probability (Week 5/6 : client-grouped, the honest version)
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier

df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)
features = ["days_since_last_update", "impressions_90d", "avg_position", "word_count", "search_volume"]

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(df, groups=df["client_id"]))
train_df, test_df = df.iloc[train_idx].copy(), df.iloc[test_idx].copy()

X_train, y_train = train_df[features].fillna(0), train_df["is_declining_label"]
model = RandomForestClassifier(n_estimators=200, max_depth=6, random_state=42)
model.fit(X_train, y_train)

df["model_prob"] = model.predict_proba(df[features].fillna(0))[:, 1]

# Final blended score (baseline still leads, since it validated better in Week 5/6)
df["final_score"] = 0.70 * df["baseline_score"].rank(pct=True) + 0.30 * df["model_prob"]

# Reason codes
def assign_reason(row):
    if stale[row.name] and visible[row.name]:
        return "stale_visible_page"
    elif row["model_prob"] >= 0.6:
        return "model_flagged_risk"
    else:
        return "low_priority"

df["reason_code"] = df.apply(assign_reason, axis=1)

# Action label
def assign_action(row):
    if row["reason_code"] == "stale_visible_page":
        return "review_for_refresh"
    elif row["reason_code"] == "model_flagged_risk":
        return "monitor"
    else:
        return "no_action"

df["action"] = df.apply(assign_action, axis=1)

queue = df.sort_values("final_score", ascending=False)[
    ["content_id", "client_id", "final_score", "reason_code", "action",
     "days_since_last_update", "impressions_90d", "avg_position", "model_prob"]
]
queue.head(20)                   

,content_id,client_id,final_score,reason_code,action,days_since_last_update,impressions_90d,avg_position,model_prob
7452,content_72496874f806,client_4ec9599fc2,0.937726,stale_visible_page,review_for_refresh,301,821,5.8,0.793508
26840,content_7f116ae1f6f5,client_9400f1b21c,0.934155,stale_visible_page,review_for_refresh,301,954,9.0,0.781451
20837,content_928af3e22c80,client_7f2253d7e2,0.914126,stale_visible_page,review_for_refresh,193,1697,15.8,0.714455
22872,content_e3ff1b093148,client_d029fa3a95,0.913265,stale_visible_page,review_for_refresh,183,1408,7.8,0.711661
26799,content_77d4d5930e5e,client_7f2253d7e2,0.912915,stale_visible_page,review_for_refresh,194,828,18.6,0.710729
11630,content_6226ee6adc91,client_d029fa3a95,0.912869,stale_visible_page,review_for_refresh,183,545,17.8,0.710729
5327,content_fe16a55cd13d,client_7f2253d7e2,0.912251,stale_visible_page,review_for_refresh,194,4556,16.4,0.708049
23215,content_bdbec75c1148,client_7f2253d7e2,0.912183,stale_visible_page,review_for_refresh,194,1316,21.8,0.708131
11489,content_5feee3994adb,client_7f2253d7e2,0.911250,stale_visible_page,review_for_refresh,194,7812,39.0,0.704477
26810,content_ecb6215e79fd,client_7f2253d7e2,0.911122,stale_visible_page,review_for_refresh,194,4429,25.3,0.704363


Pages are given a score by using two things:

Week 4 rule = 70% weight
Week 5 model probability = 30% weight

Pages are scored using two things: the Week 4 rule gets 70% and the Week 5 ML model gets 30%. The Week 4 rule gets more weight because it performed better in the honest client-grouped test. So, the 70/30 split is based on the test results, not a guess.

Reason Codes explain why a page got a certain priority. stale_visible_page means the page is old but still getting visibility, so it should be checked for updating. model_flagged_risk means the ML model thinks the page has a 60% or higher chance of declining, so it should be monitored. low_priority means neither method found an important problem.

Actions tell me what to do with the page. review_for_refresh means an editor should check the page and decide if it needs an update. monitor means keep watching the page but do not give it high priority yet. no_action means nothing needs to be done right now.

## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

This system helps a human editor decide which pages should be checked first. It ranks the pages based on their risk. It does not make the final decision. It also does not automatically edit, delete, or publish any page.

**What This System Can Say**

In Week 4, I found that old pages that are still visible were linked with higher decline rates in this dataset. The model ranks pages based on their estimated risk of decline.

In the honest client-grouped test, the model had a precision@50 of 0.62, while the simple baseline had a higher precision@50 of 0.70. This means the baseline was more accurate than the model in this test.

**What This System Cannot Say**

The system cannot say that updating a flagged page will definitely improve it. We would need a controlled experiment to prove this, and I do not have one.

It also cannot explain or predict Google's ranking algorithm. A model score should not be treated as 100% certain.

The model is also less reliable for pages with low visibility, as shown in the Week-5 error analysis.

Known Limits

The system was built using a 30,000-row anonymized dataset from a limited number of clients. The is_declining_label is based on the same time period as the data, so it is only a proxy, not a confirmed future result.

The precision results were also measured using only one train or test split. Because of these limits, the results show a general direction, but they may be different when the system is used on new data.

## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

Every page in the top 20 of the ranked list must be checked by an editor before any updating work starts. The system should not automatically approve any page.

The editor should check:

Is the page already planned for changes for another reason?
Does this type of content normally get updated rarely?(some content is only updated once a year)
Was the page recently updated, but the system did not record the new update date? This was a real problem found in Week 4's weak-picks review.
What Should NOT Be Automated
Do not automatically publish or edit a page just because it has a high score.
Do not permanently ignore a page just because it has a no_action label. It only means that there is no current warning signal; it does not mean the page is definitely healthy.
Do not fully trust model_prob for pages with low visibility, especially pages with less than about 60 impressions in 90 days. Week 5's error analysis showed that the model is least accurate for these pages.
Do not use this system for completely new clients or time periods that are very different from the training data without testing the model again first. For example, a brand-new client with no previous history should be re-validated before using this playbook.

## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*


We should regularly check Precision@20 and Precision@50 when new result data is available.

After 30 or more days, we can check what actually happened. Then we compare the real trend_direction with what the model predicted.

When Should We Retrain the Model?

If the new Precision@50 becomes much lower than 0.70, we should retrain the model or think again about using it.
If the clients change a lot, such as adding many new clients or removing old ones, we should run the client-grouped test again before trusting the new results.
If FlyRank changes the way it defines trend_direction, our current label will no longer be correct. In that case, we need to rebuild the process from Week 4.

Overall
This is a simple monitoring plan for the decision-support tool. It is not a complete production ML system. A human should regularly check the results. We are not using an automatic alert system to monitor everything.

## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [4]:
import os

os.makedirs("work/outputs", exist_ok=True)
os.makedirs("work/figures", exist_ok=True)

# Export the ranked queue (stays out of git — regenerated each run)
queue.to_csv("work/outputs/action_playbook_queue.csv", index=False)

# Export metrics as JSON (this SHOULD be committed — it's the receipt)
import json
metrics = {
    "baseline_precision_at_20": 0.80,
    "baseline_precision_at_50": 0.70,
    "model_precision_at_20": 0.65,
    "model_precision_at_50": 0.62,
    "naive_split_precision_at_20": 0.75,
    "naive_split_precision_at_50": 0.86,
    "note": "Model and baseline precision measured under honest client-grouped split (Week 5-6). Naive split shown for comparison only, not used in final scoring."
}
with open("work/outputs/playbook_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

print("Exported queue and metrics.")

Exported queue and metrics.


The ranked queue CSV file is created again every time we run the notebook.
This CSV is not saved in Git, on purpose, because the project's leak-guard prevents it from being added.
The metrics JSON file is saved in Git.
This JSON file is important because it contains the actual results and numbers used in this playbook.
Next time, these same numbers will also be used in the final paper.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.